# 07. Consolidação Final — Dataset para Modelagem

Integra todos os datasets produzidos nos notebooks anteriores numa única tabela tabular,
pronta para treino e validação de modelos de estimativa de velocidade de propagação.

| Notebook | Dados adicionados |
|---|---|
| 01 | focos INPE (base), bioma, FRP, risco_fogo |
| 03 | classe e grupo MapBiomas (cobertura do solo) |
| 04 | altitude_m (SRTM 30m) |
| 05 | wind_speed, temp_c, rh, ffwi (ERA5 por foco) |
| 06 | evento_id, velocidade_ms (variável-alvo) |

**Saída:** `data/dataset_final_2023.parquet` (+ `.csv` de fallback)

## 0. Dependências e configuração

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

try:
    import pyarrow
    PARQUET_OK = True
except ImportError:
    try:
        import fastparquet
        PARQUET_OK = True
    except ImportError:
        PARQUET_OK = False
        print('[aviso] instale: pip install pyarrow  (necessario para .parquet)')

FOCOS_DIR  = Path('data/inpe_queimadas')
MAPB_DIR   = Path('data/mapbiomas')
RELEVO_DIR = Path('data/relevo')
ERA5_DIR   = Path('data/era5')
VEL_DIR    = Path('data/velocidade')
OUT_DIR    = Path('data')
ANO        = 2023

print('Configuracao OK')
print(f'pyarrow disponivel: {PARQUET_OK}')

## 1. Carregamento e merge de todos os datasets

In [ ]:
print('Carregando e mesclando datasets...')

# Base: INPE focos 
COLS_BASE = ['latitude', 'longitude', 'data_pas', 'data_hora_gmt',
             'bioma', 'frp', 'risco_fogo', 'numero_dias_sem_chuva', 'mes']
df = pd.read_csv(
    FOCOS_DIR / f'inpe_focos_{ANO}_consolidado.csv',
    usecols=COLS_BASE,
    parse_dates=['data_pas', 'data_hora_gmt'],
    low_memory=False,
)
df['data_pas'] = pd.to_datetime(df['data_pas']).dt.normalize()
print(f'  INPE base          : {len(df):,} focos x {df.shape[1]} colunas')

# MapBiomas (nb03) 
mapb_csv = MAPB_DIR / f'focos_mapbiomas_{ANO}.csv'
if mapb_csv.exists():
    df_mapb = pd.read_csv(mapb_csv, usecols=['latitude', 'longitude',
                                              'classe_mb', 'grupo_mb'])
    df = df.merge(df_mapb, on=['latitude', 'longitude'], how='left')
    n = df['classe_mb'].notna().sum()
    print(f'  MapBiomas (nb03)   : {n:,}/{len(df):,} focos ({100*n/len(df):.0f}%)')
else:
    print(f'  [aviso] MapBiomas nao encontrado — execute nb03')

# SRTM altitude (nb04 secao 6)
alt_full = RELEVO_DIR / f'focos_altitude_completo_{ANO}.csv'
alt_samp = RELEVO_DIR / f'focos_altitude_{ANO}.csv'
for alt_csv, label in [(alt_full, 'completo'), (alt_samp, 'amostra')]:
    if alt_csv.exists():
        df_alt = pd.read_csv(alt_csv, usecols=['latitude', 'longitude', 'altitude_m'])
        df = df.merge(df_alt, on=['latitude', 'longitude'], how='left')
        n = df['altitude_m'].notna().sum()
        print(f'  SRTM ({label:<8}) (nb04): {n:,}/{len(df):,} focos ({100*n/len(df):.0f}%)')
        break
else:
    print(f'  [aviso] SRTM nao encontrado — execute nb04 (secao 6)')

# ERA5 por foco (nb05) 
era5_csv = ERA5_DIR / f'focos_era5_{ANO}.csv'
if era5_csv.exists():
    df_era5 = pd.read_csv(era5_csv,
                          usecols=['latitude', 'longitude', 'data_pas',
                                   'wind_speed_era5', 'temp_c_era5',
                                   'rh_era5', 'ffwi_era5'],
                          parse_dates=['data_pas'])
    df_era5['data_pas'] = pd.to_datetime(df_era5['data_pas']).dt.normalize()
    df = df.merge(df_era5, on=['latitude', 'longitude', 'data_pas'], how='left')
    n = df['wind_speed_era5'].notna().sum()
    print(f'  ERA5 por foco (nb05): {n:,}/{len(df):,} focos ({100*n/len(df):.0f}%)')
else:
    print(f'  [aviso] ERA5 por foco nao encontrado — execute nb05')

# Velocidade (nb06) 
vel_csv = VEL_DIR / f'focos_velocidade_{ANO}.csv'
if vel_csv.exists():
    df_vel = pd.read_csv(vel_csv,
                         usecols=['latitude', 'longitude', 'data_pas',
                                  'evento_id', 'velocidade_ms', 'velocidade_kmh'],
                         parse_dates=['data_pas'])
    df_vel['data_pas'] = pd.to_datetime(df_vel['data_pas']).dt.normalize()
    df = df.merge(df_vel, on=['latitude', 'longitude', 'data_pas'], how='left')
    n = df['velocidade_ms'].notna().sum()
    print(f'  Velocidade    (nb06): {n:,}/{len(df):,} focos ({100*n/len(df):.0f}%)')
else:
    print(f'  [aviso] Velocidade nao encontrada — execute nb06')

print(f'\nDataset final: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

## 2. Verificação do schema

In [ ]:
print('Schema do dataset final:')
print(f'  {"Coluna":<30} {"Tipo":<15} {"Preenchido":>12}  {"(%)"}')
print(f'  {"-"*65}')
for col in df.columns:
    n_ok  = df[col].notna().sum()
    pct   = 100 * n_ok / len(df)
    dtype = str(df[col].dtype)
    flag  = '' if pct > 50 else '  <- baixo'
    print(f'  {col:<30} {dtype:<15} {n_ok:>12,}  ({pct:5.1f}%){flag}')

In [ ]:
# Estatisticas das colunas numericas
cols_num = [c for c in df.columns
            if df[c].dtype in ['float64', 'int64', 'float32']
            and c not in ['mes', 'evento_id']]
print('Estatisticas das colunas numericas:')
print(df[cols_num].describe().round(2).to_string())

## 3. Exportação

In [ ]:
out_parquet = OUT_DIR / f'dataset_final_{ANO}.parquet'
out_csv     = OUT_DIR / f'dataset_final_{ANO}.csv'

if PARQUET_OK:
    df.to_parquet(out_parquet, index=False)
    print(f'Parquet salvo : {out_parquet}')
    print(f'Tamanho       : {out_parquet.stat().st_size / 1e6:.1f} MB')
else:
    df.to_csv(out_csv, index=False)
    print(f'CSV salvo     : {out_csv}')
    print(f'Tamanho       : {out_csv.stat().st_size / 1e6:.1f} MB')
    print('[info] Para parquet: pip install pyarrow')

print(f'\nColunas ({df.shape[1]}): {df.columns.tolist()}')
print(f'Focos totais          : {len(df):,}')
if 'velocidade_ms' in df.columns:
    n_alvo = df['velocidade_ms'].notna().sum()
    print(f'Focos com velocidade  : {n_alvo:,}  ({100*n_alvo/len(df):.1f}%)')
    print(f'  -> usaveis para treino supervisionado')